# Reference data: vector taxon → `vector_group`

Build `data/external/vector_taxon_groups.csv` from the VectorNet `scientificName` inventory (spec `04`).

| Column | Meaning |
|--------|---------|
| `scientific_name` | Exact VectorNet `scientificName` string |
| `vector_group` | One of `mosquito`, `tick`, `biting_midge`, `sandfly` |
| `notes` | Genus-level rule (and any QA flags) |

**Rules:** map each distinct name via its genus into the same four groups used by `VIRUS_VECTOR_TAXON_MAP`. Leave unmapped taxa out of the join (do not invent a fifth group).

In [6]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_reference_dir,
    find_repo_root,
    load_vector_detections,
)

REPO_ROOT = find_repo_root(Path.cwd())
REF_DIR = default_reference_dir(REPO_ROOT)
REF_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = REF_DIR / "vector_taxon_groups.csv"
ALLOWED_GROUPS = ("mosquito", "tick", "biting_midge", "sandfly")

REPO_ROOT, OUT_PATH

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/external/vector_taxon_groups.csv'))

## Genus → `vector_group` rules

Aligned with VectorNet taxonomic groups (Culicidae, Ixodida, Ceratopogonidae / *Culicoides*, Phlebotominae) and virus taxonomy in spec `02`.

In [7]:
# Exact genus token (first whitespace-separated word of scientificName)
GENUS_TO_VECTOR_GROUP: dict[str, str] = {
    # Mosquitoes (Culicidae)
    "Aedes": "mosquito",
    "Anopheles": "mosquito",
    "Coquillettidia": "mosquito",
    "Culex": "mosquito",
    "Culiseta": "mosquito",
    "Orthopodomyia": "mosquito",
    "Uranotaenia": "mosquito",
    # Ticks (Ixodida)
    "Dermacentor": "tick",
    "Hyalomma": "tick",
    "Ixodes": "tick",
    "Ornithodoros": "tick",
    "Rhipicephalus": "tick",
    # Biting midges (Ceratopogonidae)
    "Culicoides": "biting_midge",
    # Sandflies (Phlebotominae)
    "Phlebotomus": "sandfly",
    "Sergentomyia": "sandfly",
}

assert set(GENUS_TO_VECTOR_GROUP.values()) <= set(ALLOWED_GROUPS)
pd.Series(GENUS_TO_VECTOR_GROUP, name="vector_group").rename_axis("genus").to_frame()

,vector_group
genus,
Aedes,mosquito
Anopheles,mosquito
Coquillettidia,mosquito
Culex,mosquito
Culiseta,mosquito
Orthopodomyia,mosquito
Uranotaenia,mosquito
Dermacentor,tick
Hyalomma,tick


## Inventory VectorNet `scientificName` values

In [8]:
vectors = load_vector_detections()
assert "scientificName" in vectors.columns, vectors.columns.tolist()

name_counts = (
    vectors["scientificName"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .value_counts()
    .rename("n_rows")
    .to_frame()
    .reset_index(names="scientific_name")
    .sort_values("scientific_name")
    .reset_index(drop=True)
)

name_counts["genus"] = name_counts["scientific_name"].str.split().str[0]
print(f"Distinct scientificName: {len(name_counts):,}")
print(f"Distinct genera: {name_counts['genus'].nunique():,}")
display(name_counts.groupby("genus", as_index=False)["n_rows"].sum().sort_values("n_rows", ascending=False))
display(name_counts.head(10))

Distinct scientificName: 224
Distinct genera: 15


,genus,n_rows
4,Culicoides,48622
0,Aedes,19584
3,Culex,17477
11,Phlebotomus,16745
1,Anopheles,5857
8,Ixodes,3717
5,Culiseta,3704
13,Sergentomyia,2182
2,Coquillettidia,1049
6,Dermacentor,533


,scientific_name,n_rows,genus
0,Aedes,2071,Aedes
1,Aedes aegypti,1,Aedes
2,Aedes albopictus,6538,Aedes
3,Aedes annulipes,87,Aedes
4,Aedes behningi,1,Aedes
5,Aedes berlandi,19,Aedes
6,Aedes cantans,181,Aedes
7,Aedes caspius,3282,Aedes
8,Aedes cataphylla,34,Aedes
9,Aedes cinereus,255,Aedes


## Map names and write CSV

In [9]:
def map_vector_group(scientific_name: str) -> tuple[str | None, str]:
    genus = scientific_name.split()[0] if scientific_name else ""
    group = GENUS_TO_VECTOR_GROUP.get(genus)
    if group is None:
        return None, "unmapped_genus"
    return group, f"genus_rule:{genus}"


mapped = name_counts.copy()
mapped[["vector_group", "notes"]] = mapped["scientific_name"].apply(
    lambda n: pd.Series(map_vector_group(n))
)

unmapped = mapped[mapped["vector_group"].isna()].copy()
mapped_ok = mapped[mapped["vector_group"].notna()].copy()

assert mapped_ok["vector_group"].isin(ALLOWED_GROUPS).all()
assert not mapped_ok["scientific_name"].duplicated().any()

out = mapped_ok[["scientific_name", "vector_group", "notes"]].sort_values(
    ["vector_group", "scientific_name"]
).reset_index(drop=True)

out.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
print("By vector_group:")
display(
    out["vector_group"]
    .value_counts()
    .rename("n_taxa")
    .to_frame()
    .assign(
        n_rows=lambda x: [
            int(mapped_ok.loc[mapped_ok["vector_group"] == g, "n_rows"].sum())
            for g in x.index
        ]
    )
)

coverage = mapped_ok["n_rows"].sum() / name_counts["n_rows"].sum()
print(f"Row coverage: {mapped_ok['n_rows'].sum():,} / {name_counts['n_rows'].sum():,} ({100 * coverage:.2f}%)")
print(f"Unmapped taxa: {len(unmapped):,}")
if len(unmapped):
    display(unmapped[["scientific_name", "genus", "n_rows", "notes"]])
else:
    print("All VectorNet scientificName values mapped.")

display(out.head(15))
display(out.tail(10))

Wrote /workspaces/vector-guard/data/external/vector_taxon_groups.csv (224 rows)
By vector_group:


,n_taxa,n_rows
vector_group,,
mosquito,97,48094
biting_midge,81,48622
sandfly,39,18927
tick,7,4515


Row coverage: 120,158 / 120,158 (100.00%)
Unmapped taxa: 0
All VectorNet scientificName values mapped.


,scientific_name,vector_group,notes
0,Culicoides,biting_midge,genus_rule:Culicoides
1,Culicoides achrayi,biting_midge,genus_rule:Culicoides
2,Culicoides alazanicus,biting_midge,genus_rule:Culicoides
3,Culicoides albicans,biting_midge,genus_rule:Culicoides
4,Culicoides albihalteratus,biting_midge,genus_rule:Culicoides
5,Culicoides azerbajdzhanicus,biting_midge,genus_rule:Culicoides
6,Culicoides begueti,biting_midge,genus_rule:Culicoides
7,Culicoides brunnicans,biting_midge,genus_rule:Culicoides
8,Culicoides bysta,biting_midge,genus_rule:Culicoides
9,Culicoides cameroni,biting_midge,genus_rule:Culicoides


,scientific_name,vector_group,notes
214,Sergentomyia fallax,sandfly,genus_rule:Sergentomyia
215,Sergentomyia minuta,sandfly,genus_rule:Sergentomyia
216,Sergentomyia pawlowsky,sandfly,genus_rule:Sergentomyia
217,Dermacentor reticulatus,tick,genus_rule:Dermacentor
218,Hyalomma lusitanicum,tick,genus_rule:Hyalomma
219,Hyalomma marginatum,tick,genus_rule:Hyalomma
220,Ixodes persulcatus,tick,genus_rule:Ixodes
221,Ixodes ricinus,tick,genus_rule:Ixodes
222,Ornithodoros erraticus,tick,genus_rule:Ornithodoros
223,Rhipicephalus,tick,genus_rule:Rhipicephalus


## Sanity checks

In [10]:
reloaded = pd.read_csv(OUT_PATH, keep_default_na=False)
assert list(reloaded.columns) == ["scientific_name", "vector_group", "notes"]
assert set(reloaded["vector_group"]) <= set(ALLOWED_GROUPS)
assert not reloaded["scientific_name"].duplicated().any()

# Every observed VectorNet name should join
join = name_counts.merge(reloaded, on="scientific_name", how="left")
missing = join[join["vector_group"] == ""]
assert missing.empty, missing.head()

print("OK — vector_taxon_groups.csv ready for 08_clean_vectors / merge.")
print(reloaded.groupby("vector_group").size().to_dict())

OK — vector_taxon_groups.csv ready for 08_clean_vectors / merge.
{'biting_midge': 81, 'mosquito': 97, 'sandfly': 39, 'tick': 7}
